<div style="text-align:center; line-height:1.4; border:2px solid #6C8196; border-radius:12px; padding:20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">
  <div style="color:#6C8196; font-size:32px; font-weight:700; letter-spacing:0.4px; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:6px;">
    Final Project: Interconnect Churn Prediction
  </div>
  <br><br>
  <div style="color:#0D0A53; font-size:18px; font-weight:600;">
    Planning Phase: Data Understanding, Exploratory Analysis, and Work Plan
  </div>
</div>

<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Introduction
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:10px;">
    The goal of this project is to help the telecom operator Interconnect predict customer churn in advance. If customers who are likely to leave can be identified early, the company can respond with promotional offers, revised plan options, or service incentives designed to improve retention. From a business perspective, this makes churn prediction a 
    <span style="color:#2C5839; font-weight:600;">high-value classification task</span>
    because retaining an existing customer is often more cost-effective than acquiring a new one.
    <br><br>
    The data for this project is distributed across several related tables containing contract details, personal information, internet services, and phone services. Because the information comes from different sources, an important part of the work will be to inspect data quality, verify how the tables connect through 
    <code>customerID</code>, and determine how service usage, contract structure, and payment behavior may relate to churn. Since the main evaluation metric is 
    <span style="color:#2C5839; font-weight:600;">AUC-ROC</span>,
    the final solution must not only classify customers correctly, but also rank likely churners effectively.
  </div>

</div>

<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Planning Phase Objective
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:10px;">
    In this planning phase, the objective is to build a strong foundation for the full project by first understanding the data structure, identifying potential data preparation issues, and exploring the variables that may influence customer churn. This stage will focus on exploratory data analysis, validation of the target definition, inspection of missing values and duplicates, and early observations about feature behavior across the available datasets.
    <br><br>
    At the end of this phase, the notebook will include a set of clarifying questions and a concise step-by-step work plan that outlines how the churn prediction task will be approached in the coding and reporting stages.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Observation: Customers with month-to-month contracts may show a higher tendency to churn compared with customers on longer-term agreements.
  </div>

</div>

<div style="border:2px solid #6C8196; border-radius:12px; padding:20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08); text-align:center;">

  <div style="color:#6C8196; font-size:28px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:5px; margin-bottom:10px;">
    Exploratory Data Analysis
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:12px;">
    This section focuses on understanding the structure, consistency, and business meaning of the available datasets before preprocessing and modeling. The analysis will examine dataset dimensions, column types, missing values, duplicates, join coverage, and early indicators of features that may influence customer churn.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:16px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Goal: Build a reliable customer-level analytical table by verifying data quality, identifying preparation needs, and highlighting early churn-related patterns.
  </div>

</div>

In [1]:
from IPython.display import HTML, display

display(HTML("""
<style>
    .jp-Notebook, .notebook_app, #notebook, .container {
        background-color: #EEF2F5 !important;
    }
    .jp-Cell, .cell {
        background-color: transparent !important;
    }
</style>
"""))

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

# Load datasets
contract = pd.read_csv('/datasets/final_provider/contract.csv')
personal = pd.read_csv('/datasets/final_provider/personal.csv')
internet = pd.read_csv('/datasets/final_provider/internet.csv')
phone = pd.read_csv('/datasets/final_provider/phone.csv')

# Store in a dictionary for quick inspection
datasets = {
    'contract': contract,
    'personal': personal,
    'internet': internet,
    'phone': phone
}

# Basic structural overview
for name, df in datasets.items():
    print(f'\n{"="*60}')
    print(f'{name.upper()} DATASET')
    print(f'{"="*60}')
    print('Shape:', df.shape)
    print('\nColumns:')
    print(df.columns.tolist())
    print('\nFirst 5 rows:')
    display(df.head())


CONTRACT DATASET
Shape: (7043, 8)

Columns:
['customerID', 'BeginDate', 'EndDate', 'Type', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

First 5 rows:


,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65



PERSONAL DATASET
Shape: (7043, 5)

Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents']

First 5 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents
0,7590-VHVEG,Female,0,Yes,No
1,5575-GNVDE,Male,0,No,No
2,3668-QPYBK,Male,0,No,No
3,7795-CFOCW,Male,0,No,No
4,9237-HQITU,Female,0,No,No



INTERNET DATASET
Shape: (5517, 8)

Columns:
['customerID', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

First 5 rows:


,customerID,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,7590-VHVEG,DSL,No,Yes,No,No,No,No
1,5575-GNVDE,DSL,Yes,No,Yes,No,No,No
2,3668-QPYBK,DSL,Yes,Yes,No,No,No,No
3,7795-CFOCW,DSL,Yes,No,Yes,Yes,No,No
4,9237-HQITU,Fiber optic,No,No,No,No,No,No



PHONE DATASET
Shape: (6361, 2)

Columns:
['customerID', 'MultipleLines']

First 5 rows:


,customerID,MultipleLines
0,5575-GNVDE,No
1,3668-QPYBK,No
2,9237-HQITU,No
3,9305-CDSKC,Yes
4,1452-KIOVK,Yes


<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Initial Structural Observations
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:10px;">
    The first inspection shows that the <code>contract</code> and <code>personal</code> tables each contain <span style="color:#2C5839; font-weight:600;">7043 rows</span>, suggesting that they represent the full customer population. In contrast, the <code>internet</code> table contains <span style="color:#2C5839; font-weight:600;">5517 rows</span> and the <code>phone</code> table contains <span style="color:#2C5839; font-weight:600;">6361 rows</span>, which indicates that these service tables likely include only customers subscribed to those specific services.
    <br><br>
    All datasets contain the <code>customerID</code> field, making it the clear primary key for integration. At this stage, the structure suggests that the final analytical dataset should likely be built by using the contract table as the base and joining the remaining tables through <code>customerID</code>. Before merging, it will still be necessary to verify uniqueness of customer identifiers, inspect data types, and check whether missing rows in the service tables reflect true absence of service rather than data quality problems.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Observation: The smaller sizes of the internet and phone tables already suggest that non-matching rows after merging may carry useful business meaning, such as a customer not having internet service or multiple phone lines.
  </div>

</div>

In [3]:
for name, df in datasets.items():
    print(f'\n{"="*60}')
    print(f'{name.upper()} DATA QUALITY CHECK')
    print(f'{"="*60}')
    
    print('Data types:')
    display(df.dtypes.to_frame(name='dtype'))
    
    print('\nMissing values:')
    display(df.isna().sum().to_frame(name='missing_count'))
    
    print('\nDuplicated rows:', df.duplicated().sum())
    print('Duplicated customerID:', df['customerID'].duplicated().sum())
    print('Unique customerID count:', df['customerID'].nunique())


CONTRACT DATA QUALITY CHECK
Data types:


,dtype
customerID,object
BeginDate,object
EndDate,object
Type,object
PaperlessBilling,object
PaymentMethod,object
MonthlyCharges,float64
TotalCharges,object



Missing values:


,missing_count
customerID,0
BeginDate,0
EndDate,0
Type,0
PaperlessBilling,0
PaymentMethod,0
MonthlyCharges,0
TotalCharges,0



Duplicated rows: 0
Duplicated customerID: 0
Unique customerID count: 7043

PERSONAL DATA QUALITY CHECK
Data types:


...


Duplicated rows: 0
Duplicated customerID: 0
Unique customerID count: 5517

PHONE DATA QUALITY CHECK
Data types:


,dtype
customerID,object
MultipleLines,object



Missing values:


,missing_count
customerID,0
MultipleLines,0



Duplicated rows: 0
Duplicated customerID: 0
Unique customerID count: 6361


<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Data Quality Findings
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:10px;">
    The data quality inspection shows that all four tables are structurally consistent at the customer level. Each dataset contains <span style="color:#2C5839; font-weight:600;">unique customer identifiers only</span>, with no duplicated rows and no repeated <code>customerID</code> values. This confirms that each source is currently organized as a one-row-per-customer table, which is ideal for safe merging during preprocessing.
    <br><br>
    The <code>contract</code> and <code>personal</code> tables each contain <span style="color:#2C5839; font-weight:600;">7043 unique customers</span>, suggesting complete customer coverage. The <code>internet</code> and <code>phone</code> tables contain fewer customers, which likely indicates that they only include subscribers to those services rather than representing incomplete records. This means that unmatched rows after merging may carry real business meaning and should not automatically be treated as data errors.
    <br><br>
    Several columns will require type correction before analysis. In particular, <code>BeginDate</code> and <code>EndDate</code> are stored as text even though they contain date-like information, while <code>TotalCharges</code> is also stored as text despite being a monetary variable. These columns will need further inspection to detect hidden blanks, mixed formats, and conversion requirements before modeling begins.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Observation: The datasets appear clean in structure, so the main risks now are not duplicate records but semantic issues such as mixed date formats, text-stored numeric values, and interpreting missing joins correctly after integration.
  </div>

</div>

<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Merge Readiness and Hidden Value Inspection
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:10px;">
    After confirming that the datasets are structurally clean, the next step is to test whether they are also semantically ready for integration. This includes checking for hidden blank strings in text-based columns and confirming how many customers from the main contract table appear in the internet and phone service tables.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Goal: determine whether non-matching records reflect true absence of service and identify text fields that may require cleaning before type conversion or merging.
  </div>

</div>

In [4]:
# Check for hidden blank strings in object columns
for name, df in datasets.items():
    print(f'\n{"="*60}')
    print(f'{name.upper()} BLANK STRING CHECK')
    print(f'{"="*60}')
    
    object_cols = df.select_dtypes(include='object').columns
    
    for col in object_cols:
        blank_count = (df[col].astype(str).str.strip() == '').sum()
        print(f'{col}: {blank_count}')


CONTRACT BLANK STRING CHECK
customerID: 0
BeginDate: 0
EndDate: 0
Type: 0
PaperlessBilling: 0
PaymentMethod: 0
TotalCharges: 11

PERSONAL BLANK STRING CHECK
customerID: 0
gender: 0
Partner: 0
Dependents: 0

INTERNET BLANK STRING CHECK
customerID: 0
InternetService: 0
OnlineSecurity: 0
OnlineBackup: 0
DeviceProtection: 0
TechSupport: 0
StreamingTV: 0
StreamingMovies: 0

PHONE BLANK STRING CHECK
customerID: 0
MultipleLines: 0


<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Hidden Value Findings
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:10px;">
    The blank-string inspection shows that the datasets are almost fully clean at the text level. No hidden blank values were found in the <code>personal</code>, <code>internet</code>, or <code>phone</code> tables, and the contract table is also clean across most categorical fields. However, the <code>TotalCharges</code> column contains <span style="color:#2C5839; font-weight:600;">11 blank string values</span>, even though the earlier missing-value check reported zero nulls.
    <br><br>
    This confirms that <code>TotalCharges</code> cannot yet be treated as a reliable numeric feature without additional preprocessing. The presence of blank strings suggests that this column may contain hidden non-numeric entries, likely related to customers with very short service history or no accumulated charges at the time of data extraction. Before modeling, these values will need to be converted carefully so that the column can be analyzed as a continuous variable.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Observation: <code>TotalCharges</code> is the first confirmed preprocessing issue in the project and should be treated as a priority during data preparation.
  </div>

</div>

In [5]:
# Check how many contract customers appear in each related table
print('Customers in contract:', contract['customerID'].nunique())
print('Customers also in personal:', contract['customerID'].isin(personal['customerID']).sum())
print('Customers also in internet:', contract['customerID'].isin(internet['customerID']).sum())
print('Customers also in phone:', contract['customerID'].isin(phone['customerID']).sum())

print('\nCustomers missing from personal table:', (~contract['customerID'].isin(personal['customerID'])).sum())
print('Customers missing from internet table:', (~contract['customerID'].isin(internet['customerID'])).sum())
print('Customers missing from phone table:', (~contract['customerID'].isin(phone['customerID'])).sum())

Customers in contract: 7043
Customers also in personal: 7043
Customers also in internet: 5517
Customers also in phone: 6361

Customers missing from personal table: 0
Customers missing from internet table: 1526
Customers missing from phone table: 682


<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Merge Coverage Findings
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:10px;">
    The merge coverage check confirms that the <code>personal</code> table fully aligns with the <code>contract</code> table, since all <span style="color:#2C5839; font-weight:600;">7043 customers</span> from the contract dataset are also present in the personal dataset. This means customer demographics can be joined without any loss of coverage. The service-related tables behave differently: only <span style="color:#2C5839; font-weight:600;">5517 customers</span> appear in the internet table and <span style="color:#2C5839; font-weight:600;">6361 customers</span> appear in the phone table.
    <br><br>
    As a result, <span style="color:#2C5839; font-weight:600;">1526 customers are absent from the internet table</span> and <span style="color:#2C5839; font-weight:600;">682 customers are absent from the phone table</span>. These missing matches are unlikely to represent data errors because the earlier quality checks showed no duplicated identifiers or structural inconsistencies. A more reasonable interpretation is that these customers simply did not subscribe to the corresponding service. This is an important planning insight, because after merging, the missing values in service-related columns should likely be handled as meaningful “service not used” indicators rather than treated as ordinary missing data.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Observation: the <code>contract</code> table should be used as the main base table, with left joins to the other datasets so that all customers remain in the final modeling dataset.
  </div>

</div>

In [6]:
# Inspect the target-related column
print('Unique values in EndDate:')
print(contract['EndDate'].value_counts(dropna=False))

print('\nShare of customers with EndDate == "No":')
print((contract['EndDate'] == 'No').mean())

Unique values in EndDate:
No                     5174
2019-11-01 00:00:00     485
2019-12-01 00:00:00     466
2020-01-01 00:00:00     460
2019-10-01 00:00:00     458
Name: EndDate, dtype: int64

Share of customers with EndDate == "No":
0.7346301292063041


<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Target Definition Observations
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:10px;">
    The <code>EndDate</code> column contains two kinds of values: the string <span style="color:#2C5839; font-weight:600;">"No"</span> and actual contract termination dates. This strongly suggests that <code>"No"</code> represents customers who were still active as of the dataset reference date, while dated entries represent customers whose service had already ended. Based on the observed distribution, <span style="color:#2C5839; font-weight:600;">73.46%</span> of customers have <code>EndDate = "No"</code>, meaning most customers in the dataset appear to remain active.
    <br><br>
    This creates an important planning consideration. From a business standpoint, churn prediction usually aims to identify customers who are likely to leave, which would imply that customers with an actual end date should correspond to the churned class. However, the project clarification states that the target feature is whether <code>EndDate == "No"</code>. Because this affects class interpretation, metric reporting, and the final business framing of the model, the exact label direction should be treated as a key clarifying question before the modeling phase begins.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Observation: the target is not highly imbalanced, but its semantic direction must be confirmed carefully to ensure the model predicts the intended business outcome.
  </div>

</div>

In [8]:
# Inspect key contract-related features
for col in ['Type', 'PaperlessBilling', 'PaymentMethod']:
    print(f'\n{"="*60}')
    print(f'{col.upper()} DISTRIBUTION')
    print(f'{"="*60}')
    print(contract[col].value_counts(dropna=False))
    print('\nShare:')
    print(contract[col].value_counts(normalize=True, dropna=False).round(4))


TYPE DISTRIBUTION
Month-to-month    3875
Two year          1695
One year          1473
Name: Type, dtype: int64

Share:
Month-to-month    0.5502
Two year          0.2407
One year          0.2091
Name: Type, dtype: float64

PAPERLESSBILLING DISTRIBUTION
Yes    4171
No     2872
Name: PaperlessBilling, dtype: int64

Share:
Yes    0.5922
No     0.4078
Name: PaperlessBilling, dtype: float64

PAYMENTMETHOD DISTRIBUTION
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: PaymentMethod, dtype: int64

Share:
Electronic check             0.3358
Mailed check                 0.2289
Bank transfer (automatic)    0.2192
Credit card (automatic)      0.2161
Name: PaymentMethod, dtype: float64


<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Early Contract Feature Insights
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:10px;">
    The contract-level distributions already reveal several important structural patterns in the customer base. More than half of customers, specifically <span style="color:#2C5839; font-weight:600;">55.02%</span>, are on <code>Month-to-month</code> contracts, while the remaining customers are split between <code>One year</code> and <code>Two year</code> agreements. This is a meaningful business observation because shorter contracts generally imply greater flexibility to leave the provider, making contract type a likely high-value predictor in the churn task.
    <br><br>
    The billing profile also appears informative. A majority of customers, <span style="color:#2C5839; font-weight:600;">59.22%</span>, use <code>PaperlessBilling</code>, while the most common payment method is <code>Electronic check</code> at <span style="color:#2C5839; font-weight:600;">33.58%</span>. Since billing preferences and payment methods often reflect customer habits, convenience, and commitment level, these variables may contribute meaningful signal during modeling and should be retained for further analysis.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Observation: <code>Type</code>, <code>PaperlessBilling</code>, and <code>PaymentMethod</code> already look like strong candidate predictors and should be examined later in relation to the target.
  </div>

</div>

In [9]:
# Inspect rows where TotalCharges is blank
total_blank_rows = contract[contract['TotalCharges'].astype(str).str.strip() == '']

print('Number of rows with blank TotalCharges:', len(total_blank_rows))
display(total_blank_rows)

Number of rows with blank TotalCharges: 11


,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
488,4472-LVYGI,2020-02-01,No,Two year,Yes,Bank transfer (automatic),52.55,
753,3115-CZMZD,2020-02-01,No,Two year,No,Mailed check,20.25,
936,5709-LVOEQ,2020-02-01,No,Two year,No,Mailed check,80.85,
1082,4367-NUYAO,2020-02-01,No,Two year,No,Mailed check,25.75,
1340,1371-DWPAZ,2020-02-01,No,Two year,No,Credit card (automatic),56.05,
3331,7644-OMVMY,2020-02-01,No,Two year,No,Mailed check,19.85,
3826,3213-VVOLG,2020-02-01,No,Two year,No,Mailed check,25.35,
4380,2520-SGTTA,2020-02-01,No,Two year,No,Mailed check,20.00,
5218,2923-ARZLG,2020-02-01,No,One year,Yes,Mailed check,19.70,
6670,4075-WKNIU,2020-02-01,No,Two year,No,Mailed check,73.35,


<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    TotalCharges Blank Value Interpretation
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:10px;">
    The inspection of the <code>11</code> rows with blank <code>TotalCharges</code> reveals a consistent pattern. In every case, the customer has <code>BeginDate = 2020-02-01</code> and <code>EndDate = "No"</code>, which indicates that these customers had just started service at the reference date of the dataset and therefore had not yet accumulated a total charge value.
    <br><br>
    This suggests that the blank values in <code>TotalCharges</code> do not represent random data corruption. Instead, they reflect a specific business scenario: newly activated customers whose cumulative billing amount was not yet available. This is useful for planning because it means the missingness has interpretable meaning and can likely be handled through careful numeric conversion followed by either imputation or logically consistent replacement.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Observation: the blank <code>TotalCharges</code> values are business-driven rather than arbitrary, which makes this preprocessing issue easier to justify and handle later.
  </div>

</div>

In [10]:
# Inspect key personal features
for col in ['gender', 'SeniorCitizen', 'Partner', 'Dependents']:
    print(f'\n{"="*60}')
    print(f'{col.upper()} DISTRIBUTION')
    print(f'{"="*60}')
    print(personal[col].value_counts(dropna=False))
    print('\nShare:')
    print(personal[col].value_counts(normalize=True, dropna=False).round(4))


GENDER DISTRIBUTION
Male      3555
Female    3488
Name: gender, dtype: int64

Share:
Male      0.5048
Female    0.4952
Name: gender, dtype: float64

SENIORCITIZEN DISTRIBUTION
0    5901
1    1142
Name: SeniorCitizen, dtype: int64

Share:
0    0.8379
1    0.1621
Name: SeniorCitizen, dtype: float64

PARTNER DISTRIBUTION
No     3641
Yes    3402
Name: Partner, dtype: int64

Share:
No     0.517
Yes    0.483
Name: Partner, dtype: float64

DEPENDENTS DISTRIBUTION
No     4933
Yes    2110
Name: Dependents, dtype: int64

Share:
No     0.7004
Yes    0.2996
Name: Dependents, dtype: float64


<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Personal Feature Observations
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:10px;">
    The personal characteristics appear structurally balanced in some areas and concentrated in others. Gender is almost evenly split, with <span style="color:#2C5839; font-weight:600;">50.48%</span> male and <span style="color:#2C5839; font-weight:600;">49.52%</span> female customers, suggesting that this variable is unlikely to create strong class skew on its own. In contrast, <code>SeniorCitizen</code> is more unevenly distributed, with only <span style="color:#2C5839; font-weight:600;">16.21%</span> of customers belonging to the senior category.
    <br><br>
    Household-related variables may be more informative from a behavioral perspective. Customers without partners account for <span style="color:#2C5839; font-weight:600;">51.70%</span> of the dataset, while customers without dependents account for <span style="color:#2C5839; font-weight:600;">70.04%</span>. These features may prove useful later because household structure can influence contract stability, service usage patterns, and customer retention behavior.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Observation: the personal features look clean and interpretable, and they should be retained as potentially useful supporting predictors rather than treated as dominant drivers by default.
  </div>

</div>

In [11]:
# Inspect key internet and phone service features
service_columns = [
    'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies'
]

for col in service_columns:
    print(f'\n{"="*60}')
    print(f'{col.upper()} DISTRIBUTION')
    print(f'{"="*60}')
    print(internet[col].value_counts(dropna=False))
    print('\nShare:')
    print(internet[col].value_counts(normalize=True, dropna=False).round(4))

print(f'\n{"="*60}')
print('MULTIPLELINES DISTRIBUTION')
print(f'{"="*60}')
print(phone['MultipleLines'].value_counts(dropna=False))
print('\nShare:')
print(phone['MultipleLines'].value_counts(normalize=True, dropna=False).round(4))


INTERNETSERVICE DISTRIBUTION
Fiber optic    3096
DSL            2421
Name: InternetService, dtype: int64

Share:
Fiber optic    0.5612
DSL            0.4388
Name: InternetService, dtype: float64

ONLINESECURITY DISTRIBUTION
No     3498
Yes    2019
Name: OnlineSecurity, dtype: int64

Share:
No     0.634
Yes    0.366
Name: OnlineSecurity, dtype: float64

ONLINEBACKUP DISTRIBUTION
No     3088
Yes    2429
Name: OnlineBackup, dtype: int64

Share:
No     0.5597
Yes    0.4403
Name: OnlineBackup, dtype: float64

DEVICEPROTECTION DISTRIBUTION
No     3095
Yes    2422
Name: DeviceProtection, dtype: int64

Share:
No     0.561
Yes    0.439
Name: DeviceProtection, dtype: float64

TECHSUPPORT DISTRIBUTION
No     3473
Yes    2044
Name: TechSupport, dtype: int64

Share:
No     0.6295
Yes    0.3705
Name: TechSupport, dtype: float64

STREAMINGTV DISTRIBUTION
No     2810
Yes    2707
Name: StreamingTV, dtype: int64

Share:
No     0.5093
Yes    0.4907
Name: StreamingTV, dtype: float64

STREAMINGMOVIES DIST

<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Service Feature Observations
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:10px;">
    The service-related features show a meaningful mix of connectivity types and add-on adoption. Among internet subscribers, <span style="color:#2C5839; font-weight:600;">56.12%</span> use <code>Fiber optic</code> while <span style="color:#2C5839; font-weight:600;">43.88%</span> use <code>DSL</code>. Security and support-related services are less common, with <code>OnlineSecurity</code> and <code>TechSupport</code> each used by roughly a little more than one-third of internet customers. In contrast, entertainment-oriented services such as <code>StreamingTV</code> and <code>StreamingMovies</code> are close to evenly split between users and non-users.
    <br><br>
    The phone dataset also shows a balanced pattern, with <span style="color:#2C5839; font-weight:600;">46.71%</span> of customers having multiple lines and <span style="color:#2C5839; font-weight:600;">53.29%</span> not having them. Overall, these variables appear behaviorally rich and may help capture differences in service dependency, technical engagement, and lifestyle-related usage patterns. Since the internet and phone tables do not cover the full customer base, these features will be especially important to interpret carefully after merging, where absence from a table will likely correspond to lack of that service.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Observation: internet type, technical support services, and bundled entertainment usage all look like promising feature groups for later churn analysis.
  </div>

</div>

<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Planning Notes for Preprocessing and Feature Engineering
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.7; margin-top:10px;">
    Based on the exploratory checks completed so far, the main preparation tasks are already becoming clear. The project will require careful treatment of <code>EndDate</code>, <code>BeginDate</code>, and <code>TotalCharges</code>, since these fields contain either mixed formats or values stored as text. In addition, the service tables should later be merged in a way that preserves all customers from the contract table while allowing non-matching service records to be interpreted as meaningful absence of service.
    <br><br>
    Several feature groups also stand out as promising for the modeling phase. Contract structure, billing preferences, payment method, internet type, support-related services, and household composition all appear potentially informative. It may also be helpful to derive new features later, such as customer tenure from contract dates, total number of subscribed services, and simplified binary indicators for service adoption patterns.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Observation: the planning phase has already identified both the main preprocessing risks and the most promising feature groups, which provides a strong foundation for the modeling workflow.
  </div>

</div>

<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Clarifying Questions
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.8; margin-top:10px;">
    1. The project clarification states that the target feature is <code>EndDate == "No"</code>. Should this be interpreted exactly as the positive class, even though customers with an actual end date would normally be treated as churned from a business perspective?
    <br><br>
    2. When a customer is missing from the <code>internet</code> or <code>phone</code> table, is it correct to interpret that as the customer not using that service rather than as missing information?
    <br><br>
    3. The <code>TotalCharges</code> column contains 11 blank values, all belonging to customers whose service began on the dataset reference date. Is it acceptable to convert these values during preprocessing based on that business interpretation?
    <br><br>
    4. Since the contract data is valid as of <code>February 1, 2020</code>, should tenure-related features be calculated relative to that date for customers whose <code>EndDate</code> is <code>"No"</code>?
    <br><br>
    5. Is it acceptable to create additional engineered features, such as service count or tenure, provided they are derived only from information available in the current dataset?
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Observation: these questions are aimed at confirming target interpretation, merge semantics, and preprocessing choices before the modeling phase begins.
  </div>

</div>

<div style="border:2px solid #6C8196; border-radius:12px; padding:18px 20px; margin-bottom:18px; background-color:rgba(255,255,255,0.08);">

  <div style="color:#6C8196; font-size:24px; font-weight:700; display:inline-block; border-bottom:3px solid #6C8196; padding-bottom:4px; margin-bottom:10px;">
    Rough Work Plan
  </div>

  <div style="color:#0D0A53; font-size:15px; line-height:1.8; margin-top:10px;">
    1. Prepare and integrate the data.  
    The datasets will be cleaned, key columns will be converted to appropriate types, and all tables will be merged by <code>customerID</code> into a single customer-level dataset while preserving the full contract population.
    <br><br>
    2. Perform preprocessing and target construction.  
    The target variable will be defined carefully from <code>EndDate</code>, hidden non-numeric values will be resolved, and service-related missing matches will be handled according to their business meaning.
    <br><br>
    3. Conduct exploratory analysis and feature engineering.  
    Relationships between churn and the main customer, contract, billing, and service variables will be examined, and additional features such as tenure or service-count indicators may be created to improve predictive power.
    <br><br>
    4. Train and compare classification models.  
    Several classification algorithms will be tested and evaluated primarily with <code>AUC-ROC</code> and secondarily with accuracy in order to identify the most effective churn prediction model.
    <br><br>
    5. Select the best model and summarize conclusions.  
    The final model will be interpreted in terms of both predictive quality and business usefulness, with a summary of the most important findings, preprocessing choices, and potential retention insights.
  </div>

  <div style="color:#2C5839; font-size:15px; line-height:1.7; margin-top:18px; padding-top:12px; border-top:1.5px solid #6C8196;">
    Observation: this plan keeps the workflow logically ordered from data validation to business interpretation, which is appropriate for a churn prediction project with multiple source tables.
  </div>

</div>